# 03 - TF-IDF Evaluation & Comparison

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import joblib
import json
import os
import sys
from IPython.display import display
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, hamming_loss
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../"))
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import multilabel_confusion_matrix


## 1. Load Data, Model and Thresholds

In [ ]:
aspect_cols = ['food', 'service', 'price', 'ambiance', 'miscellaneous']

X_test_tfidf = sp.load_npz(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "X_test_tfidf.npz"))
y_test = pd.read_csv(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "y_test.csv"))

classifiers_dir = os.path.join(PROJECT_ROOT, "models", "TF-IDF", "classifiers")
model = joblib.load(os.path.join(classifiers_dir, "tfidf_ovr_logreg.joblib"))

with open(os.path.join(classifiers_dir, "tfidf_metadata.json"), "r") as f:
    metadata = json.load(f)
    
optimal_thresholds = metadata['thresholds']
print("Loaded Thresholds:", optimal_thresholds)


## 2. Generate Predictions (with fallback logic)

In [ ]:
y_test_prob = model.predict_proba(X_test_tfidf)
y_test_pred = np.zeros_like(y_test_prob)

for i, col in enumerate(aspect_cols):
    y_test_pred[:, i] = (y_test_prob[:, i] >= optimal_thresholds[col]).astype(int)

# Fallback: if no class is predicted, pick the one with max probability
empty_rows = np.flatnonzero(y_test_pred.sum(axis=1) == 0)
for row in empty_rows:
    y_test_pred[row, np.argmax(y_test_prob[row])] = 1


## 3. Evaluation Metrics

In [ ]:
print("Classification Report (TF-IDF):\n")
report = classification_report(y_test, y_test_pred, target_names=aspect_cols, zero_division=0)
print(report)

metrics_dict = {
    'F1 Micro': f1_score(y_test, y_test_pred, average='micro', zero_division=0),
    'F1 Macro': f1_score(y_test, y_test_pred, average='macro', zero_division=0),
    'F1 Samples': f1_score(y_test, y_test_pred, average='samples', zero_division=0),
    'Precision Micro': precision_score(y_test, y_test_pred, average='micro', zero_division=0),
    'Recall Micro': recall_score(y_test, y_test_pred, average='micro', zero_division=0),
    'Hamming Loss': hamming_loss(y_test, y_test_pred)
}

metrics_df = pd.DataFrame([metrics_dict]).T.reset_index()
metrics_df.columns = ['Metric', 'TF-IDF Value']
display(metrics_df)

# Save metrics
metrics_df.to_csv(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "tfidf_evaluation_metrics.csv"), index=False)


### Confusion Matrices
Plotting multilabel confusion matrices to show correct and incorrect predictions side-by-side for all aspects.

In [ ]:
matrices = multilabel_confusion_matrix(y_test, y_test_pred)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for index, aspect in enumerate(aspect_cols):
    sns.heatmap(
        matrices[index],
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axes.flat[index],
    )
    axes.flat[index].set_title(f"TF-IDF: {aspect}")
    axes.flat[index].set_xlabel("Predicted")
    axes.flat[index].set_ylabel("Actual")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()


## 4. Error Analysis

In [ ]:
# Load the raw text
split_data = np.load(os.path.join(PROJECT_ROOT, "outputs", "MPNet", "data_split.npz"))
test_idx = split_data['test_idx']
df = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "Restaurant_ABSA_processed.csv"))
X_test_text = df.iloc[test_idx]['review_en'].values

def labels_to_text(row, aspect_cols):
    row = np.asarray(row).astype(bool)
    labels = np.array(aspect_cols)[row]
    return ', '.join(labels) if len(labels) > 0 else 'None'

results_df = pd.DataFrame({
    'review': X_test_text,
    'true_aspects': [labels_to_text(row, aspect_cols) for row in y_test.values],
    'predicted_aspects': [labels_to_text(row, aspect_cols) for row in y_test_pred]
})

misclassified_df = results_df[results_df['true_aspects'] != results_df['predicted_aspects']]
print(f"Total misclassified samples: {len(misclassified_df)} out of {len(results_df)}")

# Save misclassified
misclassified_df.to_csv(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "tfidf_misclassified.csv"), index=False)
display(misclassified_df.head())